# Table State: финальный инференс (раздел 15.1 контракта)

Воспроизводит сабмит `COMBO` одной командой `run.py`. Интернет выключен, GPU не нужен, время работы меньше минуты.

**Add Input**

| вход | что берётся |
|---|---|
| датасет соревнования `rascar-hack-case-2` | пакет организаторов: таблицы, кадры, `baseline.py` |
| датасет с кодом репозитория (`throddy/table-state-code` или `victimestsate/ts-code`, коммит с `configs/submissions/COMBO.yaml`) | `run.py`, `ts/`, конфиг |
| `victimestsate/ts-artifacts-dets` | `dets/ft_yolo__v1` (свой детектор, руки) |
| `throddy/ts-artifacts-ov-pseudo` | `rowfeats/ov_counts__v1` (счётчики Grounding DINO и OWLv2) |

Детекции треков B и C читаются готовыми. Как их пересчитать на GPU: `tracks/det/README.md` и `tracks/ov/README.md` (ноутбуки `tracks/det/kaggle/`, `tracks/ov/kaggle_c2_c5.ipynb`).
Слаги в коде не зашиты: входы находятся по содержимому.


In [ ]:
import hashlib, os, shutil, subprocess, sys
from pathlib import Path

CONFIG = 'configs/submissions/COMBO.yaml'
INPUT = Path(os.environ.get('TS_KAGGLE_INPUT', '/kaggle/input'))
WORKING = Path(os.environ.get('TS_KAGGLE_WORKING', '/kaggle/working'))

# 1. Данные соревнования: каталог с train_labels.csv, tables.csv и data_card.json (как в baseline.ipynb организаторов).
data = [p.parent for p in INPUT.rglob('train_labels.csv')
        if (p.parent / 'tables.csv').is_file() and (p.parent / 'data_card.json').is_file()]
assert len(data) == 1, f'Нужен ровно один каталог данных соревнования, найдено: {data}'

# 2. Код: каталог с run.py, ts/ и нужным конфигом. Копируется в рабочую папку (входы только для чтения).
code = sorted({p.parent for p in INPUT.rglob('run.py') if (p.parent / 'ts' / 'stack.py').is_file() and (p.parent / CONFIG).is_file()})
assert len(code) == 1, f'Нужен ровно один датасет с кодом, где есть {CONFIG}; найдено: {code}'
repo = WORKING / 'table-state'
shutil.rmtree(repo, ignore_errors=True)
shutil.copytree(code[0], repo, ignore=shutil.ignore_patterns('__pycache__', '.git', 'artifacts', 'data'))

# 3. Артефакты: файлы источников из конфига собираются из подключённых датасетов в одну папку.
import yaml
cfg = yaml.safe_load((repo / CONFIG).read_text(encoding='utf-8'))
artifacts = WORKING / 'artifacts'
needed = [('dets', s) for s in cfg['sources'].get('dets', []) if s != 'prod'] + [('rowfeats', s) for s in cfg['sources'].get('rowfeats', [])]
for kind, name in needed:
    for ext in ('.parquet', '.json'):
        found = sorted(p for p in INPUT.rglob(name + ext) if p.parent.name == kind)
        assert found, f'В подключённых датасетах нет {kind}/{name}{ext}'
        (artifacts / kind).mkdir(parents=True, exist_ok=True)
        shutil.copyfile(found[0], artifacts / kind / (name + ext))
        print(f'{kind}/{name}{ext}  <-  {found[0]}')

# 4. Одна команда воспроизводимого запуска.
out = WORKING / 'submission.csv'
subprocess.run([sys.executable, 'run.py', '--data', str(data[0]), '--artifacts', str(artifacts),
                '--config', CONFIG, '--out', str(out)], cwd=repo, check=True)
print('sha256', hashlib.sha256(out.read_bytes()).hexdigest())


In [ ]:
# Сверка с сохранённым сабмитом, если он есть в датасете с кодом: счётчики должны совпасть точно.
import pandas as pd
saved = repo / 'submissions' / 'COMBO.csv'
new = pd.read_csv(out)
print(new.shape, new.columns.tolist())
if saved.is_file():
    old = pd.read_csv(saved)
    assert (new.row_id == old.row_id).all()
    for column in new.columns[1:]:
        diff = (new[column] - old[column]).abs()
        print(f'{column:<9} расхождений {(diff > 1e-9).sum():>4}, макс. {diff.max():.3g}')
        assert (diff > 1e-9).sum() == 0, f'{column} не совпал с сохранённым сабмитом'
    print('Совпадает с submissions/COMBO.csv (occ_prob до 1e-9: ExtraTrees бейзлайна складывает деревья в два потока).')
else:
    print('submissions/COMBO.csv в датасете с кодом нет, сверка пропущена.')
new.head()
